# Jamii Afya Falcon export retry

The approved 136-step training run completed and selected `stock-falcon-h1` by the existing deterministic safety gate. This notebook performs export and final validation only; it does not train or select another candidate.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
BRANCH = 'research/edge35-adaptive-streaming'
RUN = REPO / 'experiments' / 'falcon-submission-export-retry-v1'

import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for the Falcon export retry')
capabilities = [tuple(torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())]
if not capabilities or any(cap < (7, 5) for cap in capabilities):
    raise RuntimeError(f'Falcon-H1 export retry requires sm75+; detected {capabilities}')
print(json.dumps({'gpu': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], 'capabilities': capabilities}, indent=2))

def run(command, *, env=None, cwd=REPO):
    merged = os.environ.copy(); merged.update(env or {})
    merged['PYTHONUNBUFFERED'] = '1'
    print('RUN', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, cwd=cwd, env=merged, check=True)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/qeinstein/adtc-llm-limited-hardware.git', str(REPO)], cwd=WORK, env={'GIT_TERMINAL_PROMPT': '0'})
else:
    run(['git', 'fetch', 'origin', BRANCH])
    run(['git', 'checkout', '-B', BRANCH, 'origin/' + BRANCH])

os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--no-build-isolation', '-q', '-r', 'requirements-falcon-production.txt'])
run([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '-q', 'mamba-ssm>=2.2.4', 'causal-conv1d>=1.5.0'])
run([sys.executable, 'scripts/verify_falcon_fast_path.py'])
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--prefer-binary', '-q', 'llama-cpp-python'])
print('FAST_MAMBA_GATE_PASS')


In [ ]:
# This is the already-recorded deterministic outcome of the completed training run.
selection = {
    'selected_candidate': 'stock-falcon-h1',
    'selected_adapter': None,
    'criterion': 'frozen_clinical_safety_gate_only',
    'source_run': 'experiments/falcon-submission-sft-v1',
}
RUN.mkdir(parents=True, exist_ok=True)
(RUN / 'selection.json').write_text(json.dumps(selection, indent=2) + '\n', encoding='utf-8')
# The completed safety gate selected stock-falcon-h1. Use TII's canonical Q4_K_M
# export of the exact pinned stock model rather than editing the non-finite SSM-A
# tensor rejected by the local quantizer. This is an identity fallback, not a new
# candidate or a training change.
from huggingface_hub import HfApi, hf_hub_download
import hashlib, shutil
EXPORT = RUN / 'export'
EXPORT.mkdir(parents=True, exist_ok=True)
SOURCE_REPO = 'tiiuae/Falcon-H1-1.5B-Deep-Instruct-GGUF'
SOURCE_FILE = 'Falcon-H1-1.5B-Deep-Instruct-Q4_K_M.gguf'
source_revision = HfApi().model_info(SOURCE_REPO, revision='main').sha
source_path = Path(hf_hub_download(repo_id=SOURCE_REPO, filename=SOURCE_FILE, revision='main'))
final_gguf = EXPORT / 'Falcon-H1-1.5B-Deep-JamiiAfya-Q4_K_M.gguf'
shutil.copyfile(source_path, final_gguf)
submission_model = REPO / 'model' / final_gguf.name
submission_model.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(final_gguf, submission_model)
digest = hashlib.sha256(final_gguf.read_bytes()).hexdigest()
export_manifest = {
    'deployment_model': str(final_gguf),
    'deployment_bytes': final_gguf.stat().st_size,
    'deployment_sha256': digest,
    'quantization': 'Q4_K_M',
    'base_model': 'tiiuae/Falcon-H1-1.5B-Deep-Instruct',
    'base_revision': 'b6648636ddc906688974282de6e7a243395f5423',
    'adapter': None,
    'adapter_commit': None,
    'training_config_sha256': 'b6a62f73a8800c2ac1d2db962d934fe25309b6cc1c90f37b4d661a49c343bfa2',
    'selected_candidate': 'stock-falcon-h1',
    'source_gguf_repo': SOURCE_REPO,
    'source_gguf_revision': source_revision,
    'source_gguf_filename': SOURCE_FILE,
}
(EXPORT / 'export_manifest.json').write_text(json.dumps(export_manifest, indent=2) + '\n', encoding='utf-8')
print(json.dumps(export_manifest, indent=2))
run([sys.executable, 'scripts/validate_falcon_submission.py', '--model', str(submission_model), '--out-dir', str(RUN / 'final-validation')])
run([sys.executable, 'scripts/evaluate_falcon_final_48q.py', '--model', str(submission_model), '--output', 'artifacts/falcon-final-eval.json', '--no-system-output', 'artifacts/falcon-final-no-system-safety.json', '--report', 'docs/research/FALCON_FINAL_EVAL_REPORT.md', '--no-system-report', 'docs/research/FALCON_FINAL_NO_SYSTEM_SAFETY.md'])
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', 'git+https://github.com/Africa-Deep-Tech-Foundation/adtc-profiler.git@ac2e137dca65ea3b09d997774f17dd8907b489fb'])
run(['bash', 'scripts/build_llamacpp_scalar.sh'])
profiler_env = {'PATH': str(REPO / 'llama.cpp' / 'build-scalar' / 'bin') + ':' + os.environ.get('PATH', '')}
run(['adtc-profiler', 'run', '--submission', str(REPO), '--mode', 'participant', '--output', 'artifacts/adtc-submission.json'], env=profiler_env)
profiler = json.loads((REPO / 'artifacts/adtc-submission.json').read_text())
print(json.dumps({'accuracy': profiler.get('accuracy'), 'throughput': profiler.get('throughput'), 'memory': profiler.get('memory'), 'cpu_thermal': profiler.get('cpu_thermal'), 'overall_score': profiler.get('overall_score')}, indent=2))
run([sys.executable, 'scripts/update_falcon_model_card.py', '--card', 'MODEL_CARD.md', '--evaluation', 'artifacts/falcon-final-eval.json', '--profiler', 'artifacts/adtc-submission.json', '--export', str(EXPORT / 'export_manifest.json')])
run(['bash', 'scripts/host_falcon_submission.sh', str(submission_model)])
export_manifest = json.loads((EXPORT / 'export_manifest.json').read_text())
host_repo = os.environ.get('HF_REPO_ID', 'Fluxx08/jamii-afya-falcon-h1-1.5b')
host_url = os.environ.get('MODEL_URL', f'https://huggingface.co/{host_repo}/resolve/main/{final_gguf.name}')
run(['bash', 'scripts/validate_falcon_clean_clone.sh'], env={'MODEL_URL': host_url, 'MODEL_SHA256': export_manifest['deployment_sha256']})
print('FALCON_EXPORT_RETRY_COMPLETE', submission_model)
